# Advanced Python Problems with Fully Worked Solutions: Delegating with `super()`

**Topic:** Python inheritance, parent delegation, original-instance binding, initializer chaining, property overriding, and dynamic method dispatch. The supplied lesson is the primary basis for Problems 1–8. Problems 9–19 are **clearly marked advanced extensions** covering MRO and additional realistic designs; they are not claims that those topics appeared in the supplied lesson.

**Audience:** intermediate-to-advanced Python learners. **Runtime:** Python 3.10+; only Python standard-library modules are used. **Notebook structure:** 19 challenging problems, each with a specification, reasoning, executable reference solution, and tests. Run **Kernel → Restart & Run All**; each solution defines local classes in a function to avoid collisions or dependence on previous exercises.

### The governing mental model

- `super()` does **not** mean “call my hard-coded parent.” In an instance method it starts lookup **after the class where that method was defined**, in the calling instance’s **method resolution order (MRO)**.
- Calling `super().method(...)` does **not create a parent instance**; the inherited method receives the **same original `self`**.
- `self.other_method()` inside a delegated method uses normal dynamic dispatch and may select a subclass override.
- A child with no own `__init__` inherits a suitable initializer; a child that **overrides** `__init__` must delegate explicitly if it requires inherited initialization.
- `super()` is useful outside constructors, and multiple inheritance makes cooperative delegation particularly important.

**Learning strategy:** read the problem, predict the precise result and failure modes, modify the code to try your own solution, then compare to the executable reference solution. All assert statements are intentional checks; no intentionally uncaught exceptions are left behind.


## Quick reference: read an MRO and inspect delegation

The next example is a small diagnostic you can reuse when solving the problems. `__mro__` is the **actual lookup ordering**, not simply a printed family tree.


In [1]:
class DemoRoot:
    def run(self):
        return ["DemoRoot"]

class DemoMiddle(DemoRoot):
    def run(self):
        return ["DemoMiddle", *super().run()]

class DemoLeaf(DemoMiddle):
    def run(self):
        return ["DemoLeaf", *super().run()]

print("MRO:", " -> ".join(cls.__name__ for cls in DemoLeaf.__mro__))
print("Delegation:", DemoLeaf().run())
assert DemoLeaf().run() == ["DemoLeaf", "DemoMiddle", "DemoRoot"]


MRO: DemoLeaf -> DemoMiddle -> DemoRoot -> object
Delegation: ['DemoLeaf', 'DemoMiddle', 'DemoRoot']


## Problem 01 — Extend a parent method without duplicating its logic  |  Core / hard

**Specification.** Build `Person → Student → PythonStudent`, each overriding `work()`. Each implementation must add its own contribution **before** the next inherited one. Return a string; avoid printing inside business methods. The final string must contain each contribution exactly once, in the order PythonStudent → Student → Person.

**Stretch check:** explain what happens if `Student.work()` returns only its own string without calling `super()`.


**Worked solution — reasoning and best practice.**

The class-specific implementation is responsible for only its own text and delegates the rest. A missing call interrupts this chain; an explicit `Person.work(self)` would hard-code an ancestor rather than cooperate with the actual MRO. Test an exact result, not merely substring membership.


In [2]:
def solve_problem_01():
    class Person:
        def work(self):
            return "Person works"

    class Student(Person):
        def work(self):
            return f"Student studies -> {super().work()}"

    class PythonStudent(Student):
        def work(self):
            return f"PythonStudent codes -> {super().work()}"

    actual = PythonStudent().work()
    expected = "PythonStudent codes -> Student studies -> Person works"
    assert actual == expected
    assert actual.count("Person works") == 1
    print(actual)

solve_problem_01()


PythonStudent codes -> Student studies -> Person works


## Problem 02 — Find an inherited method across an intermediate class  |  Core / hard

**Specification.** Define `Person.work()`, an intermediate `Student` with no `work()`, and `PythonStudent.work()` that delegates. Predict both the result and `PythonStudent.__mro__`. Show that lookup can pass through a class where the method is absent.

**Acceptance criterion:** assert the exact MRO names and exact output.


**Worked solution — reasoning and best practice.**

`super()` starts after `PythonStudent` and searches through `Student` to `Person`. It is not restricted to direct-base attributes. `__mro__` includes `object`, even if it does not define `work()`.


In [3]:
def solve_problem_02():
    class Person:
        def work(self):
            return "Person works"

    class Student(Person):
        pass

    class PythonStudent(Student):
        def work(self):
            return "PythonStudent codes -> " + super().work()

    names = tuple(cls.__name__ for cls in PythonStudent.__mro__)
    assert names == ("PythonStudent", "Student", "Person", "object")
    assert PythonStudent().work() == "PythonStudent codes -> Person works"
    print("MRO:", " -> ".join(names))
    print("Result:", PythonStudent().work())

solve_problem_02()


MRO: PythonStudent -> Student -> Person -> object
Result: PythonStudent codes -> Person works


## Problem 03 — Prove original-instance identity across delegated calls  |  Core / hard

**Specification.** Construct a three-level hierarchy. Each delegated `record()` must append `(class_label, identity)` to a list stored on the **original instance**. Use an instance-owned list throughout the chain so no mutable class variable is accidentally shared. Confirm all identity numbers equal `id(obj)` and that a second object has an independent trace.

**Question:** why is creating `Person()` and calling `Person().record()` not equivalent to `super().record()`?


**Worked solution — reasoning and best practice.**

Each call through `super()` passes through the same object and stores state in that object. A freshly created `Person` has a different identity and different instance state. The tests verify both identity and isolation between instances rather than relying on memory-address text in representations.


In [4]:
def solve_problem_03():
    class Person:
        def __init__(self):
            self.trace = []

        def record(self):
            self.trace.append(("Person", id(self)))
            return self.trace

    class Student(Person):
        def record(self):
            self.trace.append(("Student", id(self)))
            return super().record()

    class PythonStudent(Student):
        def record(self):
            self.trace.append(("PythonStudent", id(self)))
            return super().record()

    first, second = PythonStudent(), PythonStudent()
    result = first.record()
    assert [name for name, _ in result] == ["PythonStudent", "Student", "Person"]
    assert all(identity == id(first) for _, identity in result)
    assert first.trace is result and second.trace == []
    second.record()
    assert first.trace is not second.trace
    print("Delegation labels:", [name for name, _ in result])
    print("Same original instance:", all(identity == id(first) for _, identity in result))

solve_problem_03()


Delegation labels: ['PythonStudent', 'Student', 'Person']
Same original instance: True


## Problem 04 — Inherited versus overridden initialization  |  Core / hard

**Specification.** Show three cases: (A) a child with no `__init__` inherits a required parent parameter; (B) an overriding child **forgets** parent initialization and is missing the parent field; (C) a corrected child delegates, then validates and stores its own extra field. Handle invalid input as part of the tests.

**Design rule:** create complete valid instances or raise an explicit error early.


**Worked solution — reasoning and best practice.**

A missing child initializer means normal attribute lookup finds the inherited initializer. But an overridden `__init__` replaces that behavior; Python does not silently call the parent initializer in addition. The fixed child calls `super().__init__(name)` first and validates its own parameter.


In [5]:
def solve_problem_04():
    class Person:
        def __init__(self, name):
            if not isinstance(name, str) or not name.strip():
                raise ValueError("name must be nonempty")
            self.name = name

    class InheritedStudent(Person):
        pass

    class BrokenStudent(Person):
        def __init__(self, name, student_id):
            self.student_id = student_id  # Forgot to initialize Person.

    class Student(Person):
        def __init__(self, name, student_id):
            super().__init__(name)
            if not isinstance(student_id, int) or isinstance(student_id, bool) or student_id < 0:
                raise ValueError("student_id must be a nonnegative integer")
            self.student_id = student_id

    assert InheritedStudent("Alex").name == "Alex"
    assert not hasattr(BrokenStudent("Alex", 7), "name")
    instance = Student("Alex", 7)
    assert vars(instance) == {"name": "Alex", "student_id": 7}
    for args in [("", 7), ("Alex", -1), ("Alex", True)]:
        try:
            Student(*args)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Expected a ValueError for {args!r}")
    print("Fixed instance:", vars(instance))
    print("Inherited, broken, fixed, and failure cases passed")

solve_problem_04()


Fixed instance: {'name': 'Alex', 'student_id': 7}
Inherited, broken, fixed, and failure cases passed


## Problem 05 — Predict dynamic dispatch inside a parent's method  |  Core / very hard

**Specification.** `Person.execute()` must call `self.prepare()` and `self.finish()`. `Student.execute()` first delegates to `Person.execute()` through `super()`. `Student` overrides `prepare()` but not `finish()`. Predict the event sequence and assert it. Then add a grandchild overriding `finish()` without rewriting `execute()`.

**Trap:** `super().execute()` does not force calls *inside* `Person.execute()` to use `Person.prepare()`.


**Worked solution — reasoning and best practice.**

The method chosen by `super().execute()` is the parent's implementation, but it executes with the original `Student` or `GraduateStudent` as `self`. Thus its `self.prepare()` and `self.finish()` are dynamically dispatched from the object's actual type. This is an example of the template-method design pattern.


In [6]:
def solve_problem_05():
    class Person:
        def __init__(self):
            self.events = []

        def execute(self):
            self.events.append("Person.execute")
            self.prepare()
            self.finish()

        def prepare(self):
            self.events.append("Person.prepare")

        def finish(self):
            self.events.append("Person.finish")

    class Student(Person):
        def execute(self):
            self.events.append("Student.execute")
            super().execute()

        def prepare(self):
            self.events.append("Student.prepare")

    class GraduateStudent(Student):
        def finish(self):
            self.events.append("GraduateStudent.finish")

    student, graduate = Student(), GraduateStudent()
    student.execute()
    graduate.execute()
    assert student.events == ["Student.execute", "Person.execute", "Student.prepare", "Person.finish"]
    assert graduate.events == ["Student.execute", "Person.execute", "Student.prepare", "GraduateStudent.finish"]
    print("Student:", student.events)
    print("GraduateStudent:", graduate.events)

solve_problem_05()


Student: ['Student.execute', 'Person.execute', 'Student.prepare', 'Person.finish']
GraduateStudent: ['Student.execute', 'Person.execute', 'Student.prepare', 'GraduateStudent.finish']


## Problem 06 — Make a unit-circle radius read-only while delegating its getter  |  Core / very hard

**Specification.** Build a validated `Circle` with a mutable `radius`, calculated `area`, and calculated `perimeter`. Derive `UnitCircle`, initialized using `super().__init__(1)`, that exposes a **read-only** `radius` property implemented via `super().radius`. Verify valid geometry, assignment failure, and validation for the general circle.

**Important:** this exercise enforces the **public property contract**, not absolute immutability against direct modification of `_radius`.


**Worked solution — reasoning and best practice.**

Initialize the parent's validated backing field without invoking an overridable public setter from the parent constructor; that setter would be unavailable in the read-only child. The child redefines only a getter and delegates its read via `super().radius`. A property with no setter raises `AttributeError` on ordinary assignment. Reject booleans and non-finite numbers as radii to tighten the supplied lesson's validation example.


In [7]:
def solve_problem_06():
    from math import isclose, isfinite, pi
    from numbers import Real

    class Circle:
        @staticmethod
        def _validate_radius(value):
            if (isinstance(value, bool) or not isinstance(value, Real)
                    or not isfinite(value) or value <= 0):
                raise ValueError("radius must be a finite positive real number")
            return value

        def __init__(self, radius):
            self._radius = self._validate_radius(radius)

        @property
        def radius(self):
            return self._radius

        @radius.setter
        def radius(self, value):
            self._radius = self._validate_radius(value)

        @property
        def area(self):
            return pi * self.radius ** 2

        @property
        def perimeter(self):
            return 2 * pi * self.radius

    class UnitCircle(Circle):
        def __init__(self):
            super().__init__(1)

        @property
        def radius(self):
            return super().radius

    unit = UnitCircle()
    assert unit.radius == 1
    assert isclose(unit.area, pi) and isclose(unit.perimeter, 2 * pi)
    try:
        unit.radius = 2
    except AttributeError:
        pass
    else:
        raise AssertionError("UnitCircle.radius must be read-only")
    for invalid in [0, -2, True, float("nan"), float("inf"), "3"]:
        try:
            Circle(invalid)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Invalid radius accepted: {invalid!r}")
    regular = Circle(2)
    regular.radius = 3
    assert regular.area == 9 * pi
    print(f"UnitCircle: r={unit.radius}, area={unit.area:.6f}, perimeter={unit.perimeter:.6f}")
    print("Read-only and validation tests passed")

solve_problem_06()


UnitCircle: r=1, area=3.141593, perimeter=6.283185
Read-only and validation tests passed


## Problem 07 — Override a property setter without breaking cache invalidation  |  Core / expert

**Specification.** Implement a mutable circle that caches `area` and invalidates the cache when the radius changes. A subclass must count **successful** radius assignments. Its getter should delegate using `super().radius`; its setter should invoke the inherited *property setter* correctly. Test cache identity, changed area, failure atomicity, and the fact that initialization invokes the overridden setter.

**Trap:** assigning `self.radius = value` inside `radius.setter` recurses. `super().radius = value` is not the reliable way to invoke the inherited property setter; access its descriptor's `.fset` explicitly in this single-parent exercise.


**Worked solution — reasoning and best practice.**

A property is a descriptor; `Circle.radius.fset(self, value)` reuses the parent's validation/invalidation logic without duplicating it. Initialize subclass counters **before** the parent constructor, because the parent writes `self.radius` (which dynamically selects the subclass's setter). The counter changes only after successful delegation, so failed assignments preserve all state.


In [8]:
def solve_problem_07():
    from math import isfinite, pi
    from numbers import Real

    class Circle:
        def __init__(self, radius):
            self._area_cache = None
            self.radius = radius  # Deliberately invokes the subclass setter.

        @property
        def radius(self):
            return self._radius

        @radius.setter
        def radius(self, value):
            if (isinstance(value, bool) or not isinstance(value, Real)
                    or not isfinite(value) or value <= 0):
                raise ValueError("invalid radius")
            self._radius = value
            self._area_cache = None

        @property
        def area(self):
            if self._area_cache is None:
                self._area_cache = pi * self.radius ** 2
            return self._area_cache

    class AuditedCircle(Circle):
        def __init__(self, radius):
            self.successful_assignments = 0
            super().__init__(radius)

        @property
        def radius(self):
            return super().radius

        @radius.setter
        def radius(self, value):
            Circle.radius.fset(self, value)  # Explicit descriptor delegation.
            self.successful_assignments += 1

    circle = AuditedCircle(2)
    assert circle.successful_assignments == 1
    assert circle.area == 4 * pi
    cached = circle._area_cache
    assert circle.area is cached  # The cached float object is reused.
    circle.radius = 3
    assert circle.successful_assignments == 2
    assert circle._area_cache is None
    assert circle.area == 9 * pi
    snapshot = (circle.radius, circle.area, circle.successful_assignments)
    try:
        circle.radius = -1
    except ValueError:
        pass
    else:
        raise AssertionError("Negative radius must fail")
    assert (circle.radius, circle.area, circle.successful_assignments) == snapshot
    print("Final radius, area, successful writes:", *snapshot)
    print("Cache and failure-atomicity tests passed")

solve_problem_07()


Final radius, area, successful writes: 3 28.274333882308138 2
Cache and failure-atomicity tests passed


## Problem 08 — Debug parent initialization hidden behind a child constructor  |  Core / expert

**Specification.** A base class establishes an `active` flag and maintains an event log. A child defines its own constructor and then tries to call an inherited method that expects these fields. Reproduce the bug without leaving an uncaught exception, then repair it. Ensure that calling the child constructor does not initialize the parent twice.

**Acceptance criterion:** assert the exact successful log after the fix and the relevant exception from the broken version.


**Worked solution — reasoning and best practice.**

Initialization is a contract. The broken child doesn't call `super().__init__`, so base state is absent. Call the parent once in the corrected initializer, then add child state. In production, a test should detect missing initialization through observable behavior, not by depending only on implementation details.


In [9]:
def solve_problem_08():
    class Worker:
        def __init__(self):
            self.active = True
            self.events = ["Worker initialized"]

        def run(self):
            if not self.active:
                raise RuntimeError("inactive worker")
            self.events.append("Worker ran")

    class BrokenWorker(Worker):
        def __init__(self):
            self.label = "broken"  # Missing delegation.

    try:
        BrokenWorker().run()
    except AttributeError as exc:
        assert "active" in str(exc)
    else:
        raise AssertionError("The broken constructor should fail")

    class NamedWorker(Worker):
        def __init__(self, label):
            super().__init__()
            self.label = label
            self.events.append("NamedWorker initialized")

    worker = NamedWorker("alpha")
    worker.run()
    assert worker.label == "alpha"
    assert worker.events == ["Worker initialized", "NamedWorker initialized", "Worker ran"]
    assert worker.events.count("Worker initialized") == 1
    print(worker.events)

solve_problem_08()


['Worker initialized', 'NamedWorker initialized', 'Worker ran']


# Advanced extensions beyond the supplied lesson

The remaining problems expand the lesson's single-inheritance techniques into Python's cooperative multiple inheritance, descriptors, class methods, generated constructors, asynchronous methods, class-creation hooks, and an integrated design challenge. These are deliberate extensions, not examples quoted from the source.


## Problem 09 — Derive the diamond MRO and demonstrate every cooperative call  |  Extension / expert

**Specification.** Build the diamond `Bottom(Left, Right)`, both branches extending `Root`. Every class implements `run()` and calls `super().run()`, except the terminating `Root`. Predict the exact trace and prove that the root is visited once. Inspect `Bottom.__mro__`.

**Trap:** `super()` inside `Left.run()` continues to `Right.run()` in this instance's MRO, not necessarily directly to `Root.run()`.


**Worked solution — reasoning and best practice.**

Python's C3 MRO linearizes the diamond as `Bottom, Left, Right, Root, object`. The next method after `Left` is `Right` for a `Bottom` instance. This is the reason to use cooperative `super()` rather than hard-coding direct parents in a diamond.


In [10]:
def solve_problem_09():
    class Root:
        def run(self, trace):
            trace.append("Root")
            return trace

    class Left(Root):
        def run(self, trace):
            trace.append("Left")
            return super().run(trace)

    class Right(Root):
        def run(self, trace):
            trace.append("Right")
            return super().run(trace)

    class Bottom(Left, Right):
        def run(self, trace):
            trace.append("Bottom")
            return super().run(trace)

    mro = tuple(cls.__name__ for cls in Bottom.__mro__)
    trace = Bottom().run([])
    assert mro == ("Bottom", "Left", "Right", "Root", "object")
    assert trace == ["Bottom", "Left", "Right", "Root"]
    assert trace.count("Root") == 1
    print("MRO:", " -> ".join(mro))
    print("Trace:", trace)

solve_problem_09()


MRO: Bottom -> Left -> Right -> Root -> object
Trace: ['Bottom', 'Left', 'Right', 'Root']


## Problem 10 — Compose auditable processing mixins without losing return values  |  Extension / expert

**Specification.** Design a `Pipeline` with audit and normalization mixins above a concrete processing root. The audit wrapper must record before/after events; normalization must strip and casefold a string; the concrete root must return the normalized string with an `accepted:` prefix. Prove correct event ordering and preserved return values.

**Best practice:** all cooperative implementations must accept compatible call signatures and forward to `super()`.


**Worked solution — reasoning and best practice.**

The MRO decides wrapper order. A mixin can transform arguments before delegation, observe the result afterward, and return the result unchanged. The root provides the terminal method so cooperative calls don't hit a missing `object.process`. A second pipeline with reversed mixin order shows that base order is behaviorally significant.


In [11]:
def solve_problem_10():
    class Root:
        def process(self, text, events):
            events.append("root")
            return f"accepted:{text}"

    class NormalizeMixin:
        def process(self, text, events):
            events.append("normalize")
            return super().process(text.strip().casefold(), events)

    class AuditMixin:
        def process(self, text, events):
            events.append("audit:before")
            result = super().process(text, events)
            events.append("audit:after")
            return result

    class Pipeline(AuditMixin, NormalizeMixin, Root):
        pass

    events = []
    result = Pipeline().process("  PyTHon  ", events)
    assert result == "accepted:python"
    assert events == ["audit:before", "normalize", "root", "audit:after"]

    class ReorderedPipeline(NormalizeMixin, AuditMixin, Root):
        pass

    reordered_events = []
    assert ReorderedPipeline().process("  PyTHon  ", reordered_events) == result
    assert reordered_events == ["normalize", "audit:before", "root", "audit:after"]
    print("Default:", events)
    print("Reordered:", reordered_events)
    print("Returned:", result)

solve_problem_10()


Default: ['audit:before', 'normalize', 'root', 'audit:after']
Reordered: ['normalize', 'audit:before', 'root', 'audit:after']
Returned: accepted:python


## Problem 11 — Design a cooperative `__init__` chain with keyword-only contracts  |  Extension / expert

**Specification.** A `Document` inherits `UserMixin`, `PermissionMixin`, and `InitRoot`. Each layer must consume only its own named constructor argument and forward the remaining keyword arguments. The root must reject unknown parameters and call `object.__init__` correctly. Show that all required fields initialize once; test a missing argument and an unexpected argument.

**Goal:** make construction robust when more mixins are added.


**Worked solution — reasoning and best practice.**

Use keyword-only arguments in each mixin to avoid positional ambiguity. Each cooperative initializer removes its own keyword and passes `**kwargs` onward. A terminal root checks for unused arguments then calls `super().__init__()` with no keywords, since `object.__init__` accepts no extra user-defined arguments. Required-argument failures may occur after some constructors start, but no partially initialized instance is returned.


In [12]:
def solve_problem_11():
    class InitRoot:
        def __init__(self, **kwargs):
            if kwargs:
                raise TypeError(f"Unexpected constructor keys: {sorted(kwargs)}")
            self.init_steps = ["root"]
            super().__init__()

    class PermissionMixin:
        def __init__(self, *, permission, **kwargs):
            super().__init__(**kwargs)
            self.permission = permission
            self.init_steps.append("permission")

    class UserMixin:
        def __init__(self, *, user_id, **kwargs):
            super().__init__(**kwargs)
            self.user_id = user_id
            self.init_steps.append("user")

    class Document(UserMixin, PermissionMixin, InitRoot):
        def __init__(self, *, title, **kwargs):
            super().__init__(**kwargs)
            self.title = title
            self.init_steps.append("document")

    doc = Document(title="Guide", user_id=42, permission="read")
    assert (doc.title, doc.user_id, doc.permission) == ("Guide", 42, "read")
    assert doc.init_steps == ["root", "permission", "user", "document"]
    assert doc.init_steps.count("root") == 1

    for extras in [{"unknown": 5}, {}]:
        kwargs = {"title": "Guide", "user_id": 42, **extras}
        if "unknown" in extras:
            kwargs["permission"] = "read"
        try:
            Document(**kwargs)
        except TypeError:
            pass
        else:
            raise AssertionError(f"Expected TypeError for {kwargs!r}")
    print("MRO:", " -> ".join(cls.__name__ for cls in Document.__mro__))
    print("Initialization steps:", doc.init_steps)

solve_problem_11()


MRO: Document -> UserMixin -> PermissionMixin -> InitRoot -> object
Initialization steps: ['root', 'permission', 'user', 'document']


## Problem 12 — Detect and repair a mixin that breaks the cooperation chain  |  Extension / expert

**Specification.** A pipeline uses `BrokenMetricsMixin` before `AuditMixin` and `Root`. `BrokenMetricsMixin` returns without calling `super().process()`. Show that audit and root never execute. Replace it with `MetricsMixin` that delegates exactly once, preserves the result, and adds correct metric entries.

**Acceptance criterion:** prove the difference using exact traces and return values.


**Worked solution — reasoning and best practice.**

A cooperative chain is only as complete as its participants. An early return silently bypasses later behaviors. The repaired mixin records a before event, calls `super()` once, records an after event, and returns its delegate's result. Double-delegation would be another bug because it may duplicate side effects.


In [13]:
def solve_problem_12():
    class Root:
        def process(self, data, trace):
            trace.append("root")
            return data * 2

    class AuditMixin:
        def process(self, data, trace):
            trace.append("audit")
            return super().process(data, trace)

    class BrokenMetricsMixin:
        def process(self, data, trace):
            trace.append("metrics:broken")
            return -1  # Intentionally stops the cooperative method chain.

    class BrokenPipeline(BrokenMetricsMixin, AuditMixin, Root):
        pass

    trace = []
    assert BrokenPipeline().process(5, trace) == -1
    assert trace == ["metrics:broken"]

    class MetricsMixin:
        def process(self, data, trace):
            trace.append("metrics:before")
            result = super().process(data, trace)
            trace.append("metrics:after")
            return result

    class FixedPipeline(MetricsMixin, AuditMixin, Root):
        pass

    fixed_trace = []
    assert FixedPipeline().process(5, fixed_trace) == 10
    assert fixed_trace == ["metrics:before", "audit", "root", "metrics:after"]
    assert fixed_trace.count("root") == 1
    print("Broken:", trace)
    print("Fixed:", fixed_trace)

solve_problem_12()


Broken: ['metrics:broken']
Fixed: ['metrics:before', 'audit', 'root', 'metrics:after']


## Problem 13 — Use zero-argument `super()` in a class method  |  Extension / expert

**Specification.** Define `Base → Middle → Leaf` with an overridden `@classmethod describe()` at each level. Each override delegates via zero-argument `super()` and prefixes its class-specific layer. Show that the `cls` observed in `Base.describe()` is `Leaf` when invoked on `Leaf`.

**Trap:** a class method receives `cls`, not an instance `self`.


**Worked solution — reasoning and best practice.**

Zero-argument `super()` also works in ordinary `@classmethod` definitions. It starts after the defining class in the runtime class's MRO while preserving the original bound class (`Leaf`) through the calls. This parallel to original-instance binding is worth testing explicitly.


In [14]:
def solve_problem_13():
    class Base:
        @classmethod
        def describe(cls):
            return f"Base(bound={cls.__name__})"

    class Middle(Base):
        @classmethod
        def describe(cls):
            return "Middle -> " + super().describe()

    class Leaf(Middle):
        @classmethod
        def describe(cls):
            return "Leaf -> " + super().describe()

    expected = "Leaf -> Middle -> Base(bound=Leaf)"
    assert Leaf.describe() == expected
    assert Middle.describe() == "Middle -> Base(bound=Middle)"
    print(Leaf.describe())

solve_problem_13()


Leaf -> Middle -> Base(bound=Leaf)


## Problem 14 — Compare zero-argument and explicit two-argument `super`  |  Extension / expert

**Specification.** In a diamond `Combined(Left, Right)`, call a cooperative `message()` normally. For diagnostics only, call `super(Left, instance).message()` and predict why it skips `Left` and starts from `Right`. Compare the result to explicitly calling `Left.message(instance)`.

**Best practice:** prefer ordinary zero-argument `super()` inside methods. Treat manually constructed `super(Class, instance)` as an advanced explicit lookup tool, not a default inheritance pattern.


**Worked solution — reasoning and best practice.**

`super(Left, instance)` begins searching after `Left` in `type(instance).__mro__`. It is not a request to instantiate or directly invoke `Left`'s parent. An explicit call to `Left.message(instance)` enters `Left`, whose own zero-argument `super()` then continues correctly based on the `Combined` instance.


In [15]:
def solve_problem_14():
    class Root:
        def message(self):
            return "root"

    class Left(Root):
        def message(self):
            return "left -> " + super().message()

    class Right(Root):
        def message(self):
            return "right -> " + super().message()

    class Combined(Left, Right):
        pass

    item = Combined()
    normal = item.message()
    skip_left = super(Left, item).message()
    explicit = Left.message(item)
    assert normal == "left -> right -> root"
    assert skip_left == "right -> root"
    assert explicit == normal
    print("Normal:", normal)
    print("After Left:", skip_left)
    print("Explicit Left call:", explicit)

solve_problem_14()


Normal: left -> right -> root
After Left: right -> root
Explicit Left call: left -> right -> root


## Problem 15 — Handle base initialization in dataclasses correctly  |  Extension / expert

**Specification.** A regular base class `Tracked` defines `__init__` to initialize `history`. A `@dataclass` subclass defines fields `name` and `rank`. Demonstrate that its auto-generated `__init__` does not automatically call the ordinary base's initializer. Fix this using `__post_init__`, validate the fields, and test the resulting history.

**Scope:** this is about the common case of a dataclass child of a **non-dataclass** base.


**Worked solution — reasoning and best practice.**

Dataclass-generated initializers handle dataclass fields; they do not automatically invoke a non-dataclass base's `__init__`. For this shape of class hierarchy, call `super().__init__()` from `__post_init__`. Avoid blindly generalizing the same solution to every dataclass inheritance design; consider whether the base should itself be a dataclass.


In [16]:
def solve_problem_15():
    from dataclasses import dataclass

    class Tracked:
        def __init__(self):
            self.history = ["Tracked.__init__"]

        def record(self, event):
            self.history.append(event)

    @dataclass
    class BrokenRecord(Tracked):
        name: str

    broken = BrokenRecord("example")
    assert not hasattr(broken, "history")

    @dataclass
    class Record(Tracked):
        name: str
        rank: int

        def __post_init__(self):
            if not self.name or self.rank < 0:
                raise ValueError("invalid record")
            super().__init__()
            self.record("Record.__post_init__")

    instance = Record("example", 2)
    assert (instance.name, instance.rank) == ("example", 2)
    assert instance.history == ["Tracked.__init__", "Record.__post_init__"]
    instance.record("updated")
    assert instance.history[-1] == "updated"
    try:
        Record("", 2)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid record was accepted")
    print(instance)
    print("History:", instance.history)

solve_problem_15()


solve_problem_15.<locals>.Record(name='example', rank=2)
History: ['Tracked.__init__', 'Record.__post_init__', 'updated']


## Problem 16 — Delegate an asynchronous method and preserve its result  |  Extension / expert

**Specification.** Create an asynchronous `BaseService.fetch()` and a child override that awaits `super().fetch()` before post-processing the result. Record event order and return data using Jupyter's native top-level `await`. Confirm that the original object owns the event list.

**Trap:** `super().fetch()` returns a coroutine when `fetch` is declared `async`; without `await`, the base's body does not run as expected.


**Worked solution — reasoning and best practice.**

Use `await super().fetch(...)` inside the async override, so the result is the parent's returned value rather than an unawaited coroutine. In Jupyter, use top-level `await` because the kernel already has an event loop. In a standalone Python script without a running loop, an `asyncio.run(...)` entry point would instead be appropriate.


In [17]:
async def solve_problem_16():
    import asyncio

    class BaseService:
        def __init__(self):
            self.events = []

        async def fetch(self, key):
            self.events.append("base:started")
            await asyncio.sleep(0)
            self.events.append("base:done")
            return {"key": key, "value": 10}

    class CachedService(BaseService):
        async def fetch(self, key):
            self.events.append("child:before")
            response = await super().fetch(key)
            self.events.append("child:after")
            return {**response, "cached": True}

    async def demo():
        service = CachedService()
        result = await service.fetch("alpha")
        assert result == {"key": "alpha", "value": 10, "cached": True}
        assert service.events == ["child:before", "base:started", "base:done", "child:after"]
        assert isinstance(service, BaseService)
        return result, service.events

    result, events = await demo()
    print(result)
    print(events)

await solve_problem_16()


{'key': 'alpha', 'value': 10, 'cached': True}
['child:before', 'base:started', 'base:done', 'child:after']


## Problem 17 — Cooperate in `__init_subclass__` registration hooks  |  Extension / expert

**Specification.** Two mixins attach distinct metadata to each new subclass. One hook also records a registry entry. Both `__init_subclass__` methods must delegate, so a final class defined with both mixins receives both annotations. Check that duplicate registry entries are not created and that the subclass, not a parent, is registered.

**Scope:** `__init_subclass__` runs when a subclass is **defined**, not when its instances are created.


**Worked solution — reasoning and best practice.**

Class-creation hooks can cooperate through the same MRO principle as instance methods. Both hooks accept `**kwargs` and use `super().__init_subclass__(**kwargs)`; `object.__init_subclass__` terminates the chain. Registering `cls` uses the newly created subclass. Use a local registry to keep this exercise reproducible.


In [18]:
def solve_problem_17():
    registry = []

    class TagMixin:
        def __init_subclass__(cls, **kwargs):
            super().__init_subclass__(**kwargs)
            cls.tag = f"tag:{cls.__name__.lower()}"

    class RegistryMixin:
        def __init_subclass__(cls, **kwargs):
            super().__init_subclass__(**kwargs)
            cls.registered = True
            registry.append(cls)

    class Plugin(TagMixin, RegistryMixin):
        pass

    assert Plugin.tag == "tag:plugin"
    assert Plugin.registered is True
    assert registry == [Plugin]
    assert Plugin.__mro__[:3] == (Plugin, TagMixin, RegistryMixin)
    print("Registered names:", [cls.__name__ for cls in registry])
    print("Plugin metadata:", Plugin.tag, Plugin.registered)

solve_problem_17()


Registered names: ['Plugin']
Plugin metadata: tag:plugin True


## Problem 18 — Diagnose an impossible MRO and repair the class design  |  Extension / expert

**Specification.** Create two otherwise valid bases that demand mutually inconsistent ordering, then attempt to combine them. Catch the resulting class-creation `TypeError`. Supply a consistent alternative whose MRO is valid. Assert both the conflict and the corrected inheritance order.

**Reasoning challenge:** why can't Python satisfy `A before B` and `B before A` at the same time?


**Worked solution — reasoning and best practice.**

C3 linearization preserves the ordering constraints of each base. If one branch explicitly requires `A` before `B` and another requires `B` before `A`, no linear order can satisfy both; Python rejects the class at definition time. Rather than trying to manipulate `super()` to hide this, correct the inheritance graph or prefer composition when the behaviors are not naturally cooperative.


In [19]:
def solve_problem_18():
    class A:
        pass

    class B:
        pass

    class WantsAB(A, B):
        pass

    class WantsBA(B, A):
        pass

    try:
        class Impossible(WantsAB, WantsBA):
            pass
    except TypeError as exc:
        assert "method resolution order" in str(exc).lower()
        print("Expected MRO conflict:", type(exc).__name__)
    else:
        raise AssertionError("An inconsistent MRO must be rejected")

    class AlsoWantsAB(A, B):
        pass

    class Valid(WantsAB, AlsoWantsAB):
        pass

    actual = tuple(cls.__name__ for cls in Valid.__mro__)
    assert actual == ("Valid", "WantsAB", "AlsoWantsAB", "A", "B", "object")
    print("Corrected MRO:", " -> ".join(actual))

solve_problem_18()


Expected MRO conflict: TypeError
Corrected MRO: Valid -> WantsAB -> AlsoWantsAB -> A -> B -> object


## Problem 19 — Capstone: build and test a cooperative request-processing service  |  Extension / capstone

**Scenario.** Build a realistic request processor by composing a validation mixin, an audit mixin, a metrics mixin, and a terminal storage class. A subclass must also use `super().__init__` to supply base storage state. A request is a dictionary containing `id` (a nonnegative integer, excluding `bool`) and `payload` (a nonempty string).

**Requirements:**

1. Derive `RequestService(ValidationMixin, AuditMixin, MetricsMixin, Storage)`; use `super()` in every wrapper.
2. Validate inputs **before any side effects**. On invalid input, raise `ValueError`, leave storage unchanged, do not write audit events, and do not increment a metric.
3. For a valid request, store a normalized `payload` once, return an independent output dictionary, and append a two-sided audit event.
4. Count only successful storage operations. Test duplicate IDs are rejected atomically and do not count.
5. Show exact MRO, output, events, metrics, and state-isolation checks for a second service instance.

**Best practices:** minimize hard-coded parent calls, keep constructor responsibility clear, preserve return values, make failures observable, and test invariants rather than printing and eyeballing results.


**Worked solution — reasoning and best practice.**

Arrange validation outermost so malformed inputs fail before audit/metric side effects. The terminal `Storage` validates duplicate IDs **before** mutation; metrics increments only after its delegated storage call returns successfully. Audit records `before` before delegation and `after` only after a successful response. Each service owns its own `events`, `records`, and `success_count`, all created through one cooperative initialization path. The results are copies, not the internal storage dictionaries; this avoids accidental mutation through a returned object.


In [20]:
def solve_problem_19():
    class Storage:
        def __init__(self, **kwargs):
            super().__init__(**kwargs)
            self.records = {}
            self.events = []
            self.success_count = 0

        def process(self, request):
            key = request["id"]
            if key in self.records:
                raise ValueError("duplicate id")
            normalized = request["payload"].strip().casefold()
            stored = {"id": key, "payload": normalized}
            self.records[key] = stored.copy()
            return stored.copy()

    class MetricsMixin:
        def process(self, request):
            result = super().process(request)
            self.success_count += 1
            return result

    class AuditMixin:
        def process(self, request):
            self.events.append(("before", request["id"]))
            result = super().process(request)
            self.events.append(("after", request["id"]))
            return result

    class ValidationMixin:
        def process(self, request):
            if not isinstance(request, dict):
                raise ValueError("request must be a dictionary")
            if set(request) != {"id", "payload"}:
                raise ValueError("required keys: id and payload only")
            key, payload = request["id"], request["payload"]
            if isinstance(key, bool) or not isinstance(key, int) or key < 0:
                raise ValueError("id must be a nonnegative integer")
            if not isinstance(payload, str) or not payload.strip():
                raise ValueError("payload must be nonempty")
            return super().process(request)

    class RequestService(ValidationMixin, AuditMixin, MetricsMixin, Storage):
        def __init__(self, label):
            super().__init__()
            if not isinstance(label, str) or not label.strip():
                raise ValueError("label required")
            self.label = label

    service = RequestService("primary")
    assert service.__class__.__mro__ == (
        RequestService, ValidationMixin, AuditMixin, MetricsMixin, Storage, object
    )
    invalid_requests = [
        None, {}, {"id": True, "payload": "abc"},
        {"id": -1, "payload": "abc"}, {"id": 1, "payload": "   "},
        {"id": 1, "payload": "ok", "extra": 3},
    ]
    for invalid in invalid_requests:
        snapshot = (service.records.copy(), service.events.copy(), service.success_count)
        try:
            service.process(invalid)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Invalid request succeeded: {invalid!r}")
        assert (service.records, service.events, service.success_count) == snapshot

    result = service.process({"id": 7, "payload": "  HELLO  "})
    assert result == {"id": 7, "payload": "hello"}
    assert service.records == {7: {"id": 7, "payload": "hello"}}
    assert service.events == [("before", 7), ("after", 7)]
    assert service.success_count == 1
    result["payload"] = "tampered"
    assert service.records[7]["payload"] == "hello"  # Defensive copy.

    # Duplicate detection lives in Storage. The outer AuditMixin has already
    # appended a 'before' event; we explicitly test this documented behavior.
    try:
        service.process({"id": 7, "payload": "new"})
    except ValueError as exc:
        assert "duplicate" in str(exc)
    else:
        raise AssertionError("Duplicate id must fail")
    assert service.records == {7: {"id": 7, "payload": "hello"}}
    assert service.success_count == 1
    assert service.events == [("before", 7), ("after", 7), ("before", 7)]

    other = RequestService("secondary")
    assert other.records == {} and other.events == [] and other.success_count == 0
    assert other.records is not service.records
    print("MRO:", " -> ".join(cls.__name__ for cls in RequestService.__mro__))
    print("Stored:", service.records)
    print("Audit:", service.events)
    print("Successful writes:", service.success_count)
    print("Invalid inputs, duplicates, defensive copying, and instance isolation passed")

solve_problem_19()


MRO: RequestService -> ValidationMixin -> AuditMixin -> MetricsMixin -> Storage -> object
Stored: {7: {'id': 7, 'payload': 'hello'}}
Audit: [('before', 7), ('after', 7), ('before', 7)]
Successful writes: 1
Invalid inputs, duplicates, defensive copying, and instance isolation passed


## Capstone design review: a deliberate distinction about failure atomicity

The previous capstone guarantees **no audit or metric side effects for malformed requests** because validation runs first. It also guarantees duplicate IDs do not mutate **storage** or the **success metric**, because duplicate detection occurs in `Storage` and the metrics increment follows successful delegation. However, an outer audit wrapper already emits a `before` event before the duplicate is discovered; that event intentionally remains as a record of the *attempt*. This is why the duplicate test asserts the additional `before` event rather than incorrectly claiming all state is unchanged.

**Extension exercise (not solved here):** change the audit protocol to record explicit `failure` events using `try/except` or `try/finally`, or preflight duplicate IDs before the audit layer if a completely side-effect-free duplicate policy is required. State clearly whether your guarantee is about storage atomicity, metric atomicity, audit transparency, or all three.

### Final checklist for any `super()`-based design

1. Inspect `YourClass.__mro__`, particularly in multiple inheritance.
2. Keep method signatures compatible across cooperative overrides; forward necessary arguments and return values.
3. Call `super().__init__()` when overriding initialization requires inherited state, and call it at the appropriate point.
4. Distinguish `super().method()` from ordinary `self.other_method()` inside the inherited method.
5. Remember `super().property` can delegate a getter; use the inherited property's descriptor deliberately when delegating its setter.
6. Document terminal methods, validation behavior, state ownership, duplicate side effects, and expected exceptions.
7. Test exact call order, error paths, MRO, identity/state isolation, and important invariants.

**Self-check questions:** In Problem 9, why does `Left.run()` reach `Right.run()`? In Problem 7, why must `successful_assignments` exist before `super().__init__()`? In Problem 19, why does a duplicate leave a `before` audit entry but no additional success metric?
